In [1]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification

c:\Users\Ramazan\projects\turkish-ner-berturk\ner_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("unimelb-nlp/wikiann", "tr")

In [3]:
model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
label_list = dataset["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Etiketler:", label_list)

Etiketler: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [5]:
label_list = dataset["train"].features["ner_tags"].feature.names

print("=== WIKIANN İLK 20 CÜMLE İNCELEMESİ ===\n")

for i in range(20):
    tokens = dataset["train"][i]["tokens"]
    ner_tags = dataset["train"][i]["ner_tags"]
    
    labels = [label_list[tag] for tag in ner_tags]
    
    print(f"--- Cümle {i+1} ---")
    for token, label in zip(tokens, labels):
        print(f"{token:<20} -> {label}")
    print("\n")

=== WIKIANN İLK 20 CÜMLE İNCELEMESİ ===

--- Cümle 1 ---
3.lük                -> O
maçında              -> O
Slovenya             -> B-ORG
Millî                -> I-ORG
Basketbol            -> I-ORG
Takımı'nı            -> I-ORG
yendikleri           -> O
maçta                -> O
23                   -> O
sayı                 -> O
,                    -> O
6                    -> O
ribaund              -> O
,                    -> O
2                    -> O
blok                 -> O
istatistikleriyle    -> O
oynamış              -> O
ve                   -> O
12                   -> O
faul                 -> O
yaptırmıştır         -> O
.                    -> O


--- Cümle 2 ---
'                    -> O
''                   -> O
Denizlispor          -> B-ORG
''                   -> O
'                    -> O


--- Cümle 3 ---
Hami                 -> B-PER
Mandıralı            -> I-PER
36                   -> O
,                    -> O
Orhan                -> B-PER
Çıkırıkçı        

In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
                
            previous_word_idx = word_idx
            
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [7]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names 
)

print("\n--- Hizalanmış Veri Seti Yapısı ---")
print(tokenized_datasets)


--- Hizalanmış Veri Seti Yapısı ---
DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [8]:
sample = tokenized_datasets["train"][0]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
labels = sample["labels"]

print("\n=== TOKEN VE EŞLEŞEN ETIKET KONTROLÜ ===")
print(f"{'TOKEN':<20} | {'ETİKET ID':<10} | {'ETİKET ADI'}")
print("-" * 50)

for tok, lab in zip(tokens, labels):
    label_name = id2label[lab] if lab != -100 else "MASKE -100"
    print(f"{tok:<20} | {lab:<10} | {label_name}")


=== TOKEN VE EŞLEŞEN ETIKET KONTROLÜ ===
TOKEN                | ETİKET ID  | ETİKET ADI
--------------------------------------------------
[CLS]                | -100       | MASKE -100
3                    | 0          | O
.                    | -100       | MASKE -100
lük                  | -100       | MASKE -100
maçında              | 0          | O
Slovenya             | 3          | B-ORG
Millî                | 4          | I-ORG
Basketbol            | 4          | I-ORG
Takımı               | 4          | I-ORG
'                    | -100       | MASKE -100
nı                   | -100       | MASKE -100
yendi                | 0          | O
##k                  | -100       | MASKE -100
##leri               | -100       | MASKE -100
maçta                | 0          | O
23                   | 0          | O
sayı                 | 0          | O
,                    | 0          | O
6                    | 0          | O
riba                 | 0          | O
##und                

In [9]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("\nModel ve DataCollator eğitime hazır!")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3604.52it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch


Model ve DataCollator eğitime hazır!
